# MNIST Training and Deployment with PyTorch on SageMaker

<div class="alert alert-block alert-info">
⚠️ The latest SageMaker Distribution image version known to work with this notebook is <code>3.7.0</code>. If you encounter problems with other versions, please downgrade to version <code>3.7.0</code>. <b>To do so, you must stop your JupyterApp, downgrade the SageMaker Distribution image to <code>3.7.0</code> and restart the JupyterLabApp for the changes to take effect</b>.</div>

<div class="alert alert-block alert-info">
⚠️ This notebbok has been tested with SageMaker Python SDK version <code>3.4.0</code>. If you are using a different version and encounter issues, please re-install version <code>3.4.0</code> and restart the Kernel. Please make sure that you are using the <code>Python 3 (ipykernel)</code> kernel</div>

This notebook demonstrates how to train a PyTorch model on Amazon SageMaker and deploy it as a real-time inference endpoint.

**What you'll learn:**
- Use `ModelTrainer` to run distributed training jobs
- Use `ModelBuilder` to deploy models as endpoints
- Write training and inference scripts for SageMaker containers
- Invoke endpoints for real-time predictions

In [ ]:
!pip uninstall -y torchvision
!pip install -qU torchvision sagemaker==3.4.0

In [ ]:
import sagemaker
from importlib.metadata import version

print(f'sagemaker version: {version('sagemaker')}')

## Contents

1. [Background](#Background)
1. [Setup](#Setup)
1. [Data](#Data)
1. [Train](#Train)
1. [Host](#Host)

---

## Background

MNIST is a widely used dataset for handwritten digit classification. It consists of 70,000 labeled 28x28 pixel grayscale images of hand-written digits. The dataset is split into 60,000 training images and 10,000 test images. There are 10 classes (one for each of the 10 digits).

---

## Setup

We start by creating a `Session` object which manages interactions with SageMaker APIs and AWS services. The session provides:

- `default_bucket()`: S3 bucket for storing training data and model artifacts
- `upload_data()`: Upload local files to S3

We also retrieve the IAM execution role using `get_execution_role()`, which grants SageMaker permission to access AWS resources on your behalf.


In [ ]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker_session = Session()

bucket = sagemaker_session.default_bucket()
prefix = "sagemaker/pytorch-mnist"

role = get_execution_role()

## Data
### Getting the data



In [ ]:
from torchvision.datasets import MNIST
from torchvision import transforms

MNIST.mirrors = ["https://sagemaker-sample-files.s3.amazonaws.com/datasets/image/MNIST/"]

MNIST(
    "lab03-pytorch-data",
    download=True,
    transform=transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
    ),
)

### Uploading the data to S3
We are going to use the `sagemaker.Session.upload_data` function to upload our datasets to an S3 location. The return value inputs identifies the location -- we will use later when we start the training job.


In [ ]:
inputs = sagemaker_session.upload_data(path="lab03-pytorch-data", bucket=bucket, key_prefix=prefix)
print("input spec (in this case, just an S3 path): {}".format(inputs))

## Train
### Training script

The training script (`mnist-pytorch.py`) runs inside a SageMaker training container. It accesses the training environment through environment variables:

| Variable | Description |
|----------|-------------|
| `SM_MODEL_DIR` | Directory to save model artifacts (uploaded to S3 after training) |
| `SM_CHANNEL_TRAINING` | Directory containing the training data |
| `SM_NUM_GPUS` | Number of GPUs available |
| `SM_HOSTS` | List of all hosts (for distributed training) |

The script must save the trained model to `SM_MODEL_DIR`. SageMaker automatically packages and uploads it to S3.

In [ ]:
!mkdir -p mnist-pytorch/

In [ ]:
%%writefile mnist-pytorch/train.py
import argparse
import json
import logging
import os
import sys

#import sagemaker_containers
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
import torch.utils.data.distributed
from torchvision import datasets, transforms

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.addHandler(logging.StreamHandler(sys.stdout))


# Based on https://github.com/pytorch/examples/blob/master/mnist/main.py
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


def _get_train_data_loader(batch_size, training_dir, is_distributed, **kwargs):
    logger.info("Get train data loader")
    dataset = datasets.MNIST(
        training_dir,
        train=True,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    )
    train_sampler = (
        torch.utils.data.distributed.DistributedSampler(dataset) if is_distributed else None
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train_sampler is None,
        sampler=train_sampler,
        **kwargs
    )


def _get_test_data_loader(test_batch_size, training_dir, **kwargs):
    logger.info("Get test data loader")
    return torch.utils.data.DataLoader(
        datasets.MNIST(
            training_dir,
            train=False,
            transform=transforms.Compose(
                [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
            ),
        ),
        batch_size=test_batch_size,
        shuffle=True,
        **kwargs
    )


def _average_gradients(model):
    # Gradient averaging.
    size = float(dist.get_world_size())
    for param in model.parameters():
        dist.all_reduce(param.grad.data, op=dist.reduce_op.SUM)
        param.grad.data /= size


def train(args):
    is_distributed = len(args.hosts) > 1 and args.backend is not None
    logger.debug("Distributed training - {}".format(is_distributed))
    use_cuda = args.num_gpus > 0
    logger.debug("Number of gpus available - {}".format(args.num_gpus))
    kwargs = {"num_workers": 1, "pin_memory": True} if use_cuda else {}
    device = torch.device("cuda" if use_cuda else "cpu")

    if is_distributed:
        # Initialize the distributed environment.
        world_size = len(args.hosts)
        os.environ["WORLD_SIZE"] = str(world_size)
        host_rank = args.hosts.index(args.current_host)
        os.environ["RANK"] = str(host_rank)
        dist.init_process_group(backend=args.backend, rank=host_rank, world_size=world_size)
        logger.info(
            "Initialized the distributed environment: '{}' backend on {} nodes. ".format(
                args.backend, dist.get_world_size()
            )
            + "Current host rank is {}. Number of gpus: {}".format(dist.get_rank(), args.num_gpus)
        )

    # set the seed for generating random numbers
    torch.manual_seed(args.seed)
    if use_cuda:
        torch.cuda.manual_seed(args.seed)

    train_loader = _get_train_data_loader(args.batch_size, args.data_dir, is_distributed, **kwargs)
    test_loader = _get_test_data_loader(args.test_batch_size, args.data_dir, **kwargs)

    logger.debug(
        "Processes {}/{} ({:.0f}%) of train data".format(
            len(train_loader.sampler),
            len(train_loader.dataset),
            100.0 * len(train_loader.sampler) / len(train_loader.dataset),
        )
    )

    logger.debug(
        "Processes {}/{} ({:.0f}%) of test data".format(
            len(test_loader.sampler),
            len(test_loader.dataset),
            100.0 * len(test_loader.sampler) / len(test_loader.dataset),
        )
    )

    model = Net().to(device)
    if is_distributed and use_cuda:
        # multi-machine multi-gpu case
        model = torch.nn.parallel.DistributedDataParallel(model)
    else:
        # single-machine multi-gpu case or single-machine or multi-machine cpu case
        model = torch.nn.DataParallel(model)

    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum)

    for epoch in range(1, args.epochs + 1):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader, 1):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            if is_distributed and not use_cuda:
                # average gradients manually for multi-machine cpu case only
                _average_gradients(model)
            optimizer.step()
            if batch_idx % args.log_interval == 0:
                logger.info(
                    "Train Epoch: {} [{}/{} ({:.0f}%)] Loss: {:.6f}".format(
                        epoch,
                        batch_idx * len(data),
                        len(train_loader.sampler),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )
        test(model, test_loader, device)
    save_model(model, args.model_dir)


def test(model, test_loader, device):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, size_average=False).item()  # sum up batch loss
            pred = output.max(1, keepdim=True)[1]  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    logger.info(
        "Test set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n".format(
            test_loss, correct, len(test_loader.dataset), 100.0 * correct / len(test_loader.dataset)
        )
    )


def model_fn(model_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = torch.nn.DataParallel(Net())
    with open(os.path.join(model_dir, "model.pth"), "rb") as f:
        model.load_state_dict(torch.load(f))
    return model.to(device)


def save_model(model, model_dir):
    logger.info("Saving the model.")
    path = os.path.join(model_dir, "model.pth")
    # recommended way from http://pytorch.org/docs/master/notes/serialization.html
    torch.save(model.cpu().state_dict(), path)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    # Data and model checkpoints directories
    parser.add_argument(
        "--batch-size",
        type=int,
        default=64,
        metavar="N",
        help="input batch size for training (default: 64)",
    )
    parser.add_argument(
        "--test-batch-size",
        type=int,
        default=1000,
        metavar="N",
        help="input batch size for testing (default: 1000)",
    )
    parser.add_argument(
        "--epochs",
        type=int,
        default=10,
        metavar="N",
        help="number of epochs to train (default: 10)",
    )
    parser.add_argument(
        "--lr", type=float, default=0.01, metavar="LR", help="learning rate (default: 0.01)"
    )
    parser.add_argument(
        "--momentum", type=float, default=0.5, metavar="M", help="SGD momentum (default: 0.5)"
    )
    parser.add_argument("--seed", type=int, default=1, metavar="S", help="random seed (default: 1)")
    parser.add_argument(
        "--log-interval",
        type=int,
        default=100,
        metavar="N",
        help="how many batches to wait before logging training status",
    )
    parser.add_argument(
        "--backend",
        type=str,
        default=None,
        help="backend for distributed training (tcp, gloo on cpu and gloo, nccl on gpu)",
    )

    # Container environment
    parser.add_argument("--hosts", type=list, default=json.loads(os.environ["SM_HOSTS"]))
    parser.add_argument("--current-host", type=str, default=os.environ["SM_CURRENT_HOST"])
    parser.add_argument("--model-dir", type=str, default=os.environ["SM_MODEL_DIR"])
    parser.add_argument("--data-dir", type=str, default=os.environ["SM_CHANNEL_TRAINING"])
    parser.add_argument("--num-gpus", type=int, default=os.environ["SM_NUM_GPUS"])

    train(parser.parse_args())


### Run training in SageMaker

`ModelTrainer` orchestrates training jobs on SageMaker infrastructure. Key components:

- **`training_image`**: Docker image URI containing the ML framework (PyTorch, TensorFlow, etc.)
- **`SourceCode`**: Specifies your training script (`entry_script`) and its directory (`source_dir`)
- **`Compute`**: Defines instance type and count for training
- **`hyperparameters`**: Dictionary of training parameters (values must be strings)
- **`Torchrun()`**: Enables distributed training using PyTorch's torchrun launcher

**Distributed Training:** This example trains on 2 instances (`instance_count=2`) with `Torchrun()` enabled. The training script uses PyTorch's `DistributedDataParallel` to synchronize gradients across instances. Each instance receives the full dataset, and gradients are averaged during backpropagation.

Use `image_uris.retrieve()` to get the appropriate framework container image for your region and instance type.


In [ ]:
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import Compute, SourceCode
from sagemaker.core import image_uris
from sagemaker.train.distributed import Torchrun


image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_region_name,
    version="2.5",
    py_version="py311",
    instance_type="ml.m5.2xlarge",
    image_scope="training",
)

estimator = ModelTrainer(
    training_image=image_uri,
    role=role,
    source_code=SourceCode(
        source_dir="mnist-pytorch",
        entry_script="train.py"
    ),
    compute=Compute(instance_type="ml.m5.2xlarge", instance_count=2),
    hyperparameters={"epochs": "1", "backend": "gloo"},
    distributed=Torchrun(),  # Enables distributed training
    sagemaker_session=sagemaker_session,
)

Call `trainer.train()` to start the training job. The `InputData` dataclass maps a channel name to an S3 data source:

- **`channel_name`**: Name of the input channel (accessible via `SM_CHANNEL_<NAME>` in the script)
- **`data_source`**: S3 URI containing the training data

SageMaker downloads the data to the container and makes it available at the channel path.


In [ ]:
from sagemaker.core.training.configs import InputData

estimator.train(input_data_config=[InputData(channel_name="training", data_source=inputs)])

## Host
### Create endpoint

`ModelBuilder` packages your model and deploys it as a real-time inference endpoint. The deployment process involves several steps:

1. **Retrieve inference image**: Use `image_uris.retrieve()` with `image_scope="inference"` to get a serving container
2. **Define input/output schema**: `SchemaBuilder` specifies the expected tensor shapes for serialization
3. **Create inference script**: A Python script that defines how to load and run the model
4. **Build and deploy**: `ModelBuilder` repacks the model with your code and creates the endpoint

#### Inference Script Requirements

The inference script must define at minimum:
- `model_fn(model_dir)`: Load and return the model from the given directory

Optional functions for custom serialization:
- `input_fn(request_body, content_type)`: Deserialize input data
- `predict_fn(input_data, model)`: Run inference
- `output_fn(prediction, accept)`: Serialize output

**Important:** The inference script must include the model class definition (e.g., `Net`) since the saved model only contains weights, not the architecture.


In [ ]:
inference_image = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_region_name,
    version="2.5",
    py_version="py311",
    instance_type="ml.m5.2xlarge",
    image_scope="inference",  # Not "training"
)

In [ ]:
%%writefile mnist-pytorch/inference.py
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

def model_fn(model_dir):
    print(f'model_dir: {model_dir}')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = torch.nn.DataParallel(Net())
    with open(os.path.join(model_dir, "model.pth"), "rb") as f:
        model.load_state_dict(torch.load(f))
    return model.to(device)

In [ ]:
from sagemaker.serve.builder.schema_builder import SchemaBuilder
import torch

schema_builder = SchemaBuilder(
    sample_input=torch.randn(1, 1, 28, 28),  # MNIST input shape
    sample_output=torch.randn(1, 10),        # 10 classes output
)

#### Working Directory for Model Repacking

`ModelBuilder` needs a writable directory to:
1. Download model artifacts from S3
2. Extract and repack the model with your inference code
3. Upload the repacked model back to S3

In SageMaker Studio, the default temp directory may not exist or be writable. We create `/tmp/model-builder` as a working directory.


In [ ]:
!mkdir -p /tmp/model-builder

#### ModelBuilder Configuration

Key parameters:
- **`image_uri`**: Inference container with the serving framework
- **`s3_model_data_url`**: S3 path to trained model artifacts (from `trainer._latest_training_job.model_artifacts.s3_model_artifacts`)
- **`schema_builder`**: Input/output schema for request serialization
- **`source_code`**: Directory and script containing inference code
- **`model_server`**: Serving backend (TorchServe for PyTorch models)
- **`model_path`**: Local directory for model repacking operations

Call `build()` to repack the model with inference code, then `deploy()` to create the endpoint.


In [ ]:
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.utils.types import ModelServer


model_data_url = estimator._latest_training_job.model_artifacts.s3_model_artifacts

model_builder = ModelBuilder(
    image_uri=inference_image,
    s3_model_data_url=model_data_url,
    schema_builder=schema_builder,
    source_code=SourceCode(
        source_dir="mnist-pytorch",  # directory containing your script
        entry_script="inference.py"
    ),
    role_arn=role,
    model_server=ModelServer.TORCHSERVE,  # Add this to avoid Triton auto-detection
    dependencies={},
    model_path="/tmp/model-builder"
)

model_builder.build()

In [ ]:
endpoint = model_builder.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
)

### Evaluate

Send inference requests using `endpoint.invoke()`:

- **`body`**: Serialized input data (e.g., JSON string)
- **`content_type`**: MIME type of the request body

The response contains a `body` stream that must be read and decoded.

You can also use the boto3 `sagemaker-runtime` client's `invoke_endpoint()` for direct API access.


In [ ]:
!ls lab03-pytorch-data/MNIST/raw

In [ ]:
import gzip
import os
import random
import numpy as np


data_dir = "lab03-pytorch-data/MNIST/raw"
with gzip.open(os.path.join(data_dir, "t10k-images-idx3-ubyte.gz"), "rb") as f:
    images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28).astype(np.float32)

mask = random.sample(range(len(images)), 16)  # randomly select some of the test images
mask = np.array(mask, dtype=int)
data = images[mask]

In [ ]:
import json

# Invoke endpoint
input_data = np.expand_dims(data, axis=1)
response = endpoint.invoke(
    body=json.dumps(input_data.tolist()),
    content_type="application/json"
)

In [ ]:
# Parse response
result = json.loads(response.body.read().decode("utf-8"))
print("Raw prediction result:")
print(result)
print()

labeled_predictions = list(zip(range(10), result[0]))
print("Labeled predictions: ")
print(labeled_predictions)
print()

labeled_predictions.sort(key=lambda label_and_prob: 1.0 - label_and_prob[1])
print("Most likely answer: {}".format(labeled_predictions[0]))

In [ ]:
import boto3
import json

client = boto3.client("sagemaker-runtime", region_name=sagemaker_session.boto_session.region_name)

input_data = np.expand_dims(data, axis=1)
response = client.invoke_endpoint(
    EndpointName=endpoint.endpoint_name,
    ContentType="application/json",
    Body=json.dumps(input_data.tolist())
)

result = json.loads(response["Body"].read().decode("utf-8"))
print(result)

### Cleanup

Delete the endpoint when finished to stop incurring charges. The endpoint name is available via `endpoint.endpoint_name`.


In [ ]:
sagemaker_session.delete_endpoint(endpoint_name=endpoint.endpoint_name)